# Laboratorio Gradio de ruido y difusión

Este entorno es independiente de la app de Julia. Permite cargar **cualquier PNG o TIFF de 8 bits**, ver el archivo original y su referencia preparada, mover sigma para observar el ruido y aplicar difusión con un botón separado. No obliga a utilizar `potw1803a.tif` ni un hash de imagen fijo.

Selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**, si está disponible. En la interfaz deja `cuda` para inferencia; generar ruido y visualizar mapas no necesita GPU. La primera inferencia descarga/carga los pesos. Las imágenes se conservan a la resolución de trabajo seleccionada, sin superresolución.

Los mapas incluyen la escala absoluta y una vista multiplicada por una ganancia explícita. Un mapa oscuro puede contener diferencias pequeñas; la vista amplificada no altera los datos. Las estadísticas describen diferencias de píxeles, no PSNR, SSIM ni autenticidad.

In [ ]:
import os
import signal
import subprocess

def ejecutar_con_salida(comando, cwd=None):
    """Muestra stdout y stderr del proceso en la salida de la celda de VS Code."""
    entorno = os.environ.copy()
    entorno['PYTHONUNBUFFERED'] = '1'
    print('Proceso iniciado. Los mensajes aparecerán debajo de esta celda.', flush=True)
    with subprocess.Popen(
        comando, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=entorno,
        start_new_session=(os.name == 'posix'),
    ) as proceso:
        try:
            for linea in proceso.stdout:
                print(linea, end='', flush=True)
            codigo = proceso.wait()
        except KeyboardInterrupt:
            if proceso.poll() is None:
                if os.name == 'posix':
                    os.killpg(proceso.pid, signal.SIGTERM)
                else:
                    proceso.terminate()
                try:
                    proceso.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    if os.name == 'posix':
                        os.killpg(proceso.pid, signal.SIGKILL)
                    else:
                        proceso.kill()
                    proceso.wait()
            raise
    if codigo != 0:
        raise subprocess.CalledProcessError(codigo, comando)
    print('Proceso finalizado correctamente.', flush=True)


In [ ]:
from pathlib import Path
import subprocess
import sys

if not Path('/content').is_dir():
    raise RuntimeError('Este notebook está preparado para Google Colab.')
proyecto = Path('/content/astro-restauracion')
if not proyecto.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Sabri-Nang/astro-restauracion.git', str(proyecto)], check=True)
print('Revisión:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=proyecto, text=True).strip())
if not (proyecto / 'laboratorio.py').is_file():
    raise RuntimeError('Falta laboratorio.py. Publica los cambios y actualiza la copia de esta sesión con git pull --ff-only origin main; luego repite esta celda.')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.3'], check=True)
ejecutar_con_salida(['uv', 'sync', '--locked', '--python', '3.12'], cwd=proyecto)
ejecutar_con_salida(
    ['uv', 'run', '--locked', 'python', '-u', '-c',
     "import sys, torch; assert sys.version_info[:2] == (3, 12); print('CUDA disponible:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No disponible: solo vista de ruido o CPU explícita')"],
    cwd=proyecto,
)

## Abrir el laboratorio

Si ya tenías el repositorio en esta sesión, interrumpe Gradio y actualiza con `ejecutar_con_salida(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=proyecto)`. Revisa cualquier mensaje sobre cambios locales antes de continuar. Los cambios locales del proyecto deben publicarse para que el clon de Colab pueda descargarlos.

La siguiente celda mantiene el servidor activo: abre el enlace `https://…gradio.live` y no interrumpas la celda mientras uses el laboratorio. Es un enlace temporal accesible a quien lo tenga. Colab puede limitar el uso de interfaces externas; consulta https://research.google.com/colaboratory/faq.html.

1. Sube un PNG o TIFF de 8 bits; puedes cargar otra imagen en cualquier momento.
2. Abre el panel de original y procedencia para ver el archivo sin reducir e indicar fuente, crédito y licencia. La referencia de cálculo es la original convertida a RGB y ajustada de tamaño.
3. Mueve sigma y observa la copia con ruido. Se conserva la semilla para comparar intensidades. Al cambiar imagen, ruido o tamaño se descarta la salida anterior.
4. Elige el par del mapa y ajusta la ganancia visual. La imagen absoluta siempre mantiene escala 0–255.
5. Pulsa **Aplicar difusión a la copia ruidosa**. Solo ese botón carga el modelo y ejecuta inferencia. Mientras se procesa, espera antes de modificar controles; las operaciones se atienden en orden para evitar mezclar entradas.
6. Compara las tres vistas y descarga el ZIP del ensayo. La descarga incluye referencia, ruido, salida, mapas y registros; conserva también tu original. No se verifican autenticidad ni eficacia científica.

Antes de compartir archivos, comprueba las condiciones de las imágenes que subas. Para los ejemplos del proyecto consulta `assets/ejemplos/README.md`; la app no presume una licencia a partir del nombre de un archivo.

In [ ]:
ejecutar_con_salida(
    ['uv', 'run', '--locked', 'python', '-u', '-c',
     "from laboratorio import create_lab; create_lab().launch(share=True, server_name='0.0.0.0')"],
    cwd=proyecto,
)